# Step 8 Part 1: Deployment

This notebook implements the first part of the LightGBM deployment pipeline: a **prediction function** that accepts raw user and product features and returns a reorder probability, demonstrated on concrete examples drawn from the dataset.

## Deployment Plan

| Part | Content | Status |
|---|---|---|
| **Part 1 (this notebook)** | **Prediction function + example demonstrations** | **Current** |
| Part 2 | Batch scoring pipeline -- rank all products for a given user | Upcoming |
| Part 3 | REST API (Flask) -- serve predictions over HTTP | Upcoming |

## Why LightGBM
As determined in Step 7, LightGBM is the recommended model for deployment: best ROC-AUC (0.9710), best Brier Score (0.0458), and native categorical feature handling that extracts signal unavailable to the other models.

## What a Prediction Function Does
At inference time, the model does not have access to the pre-computed feature tables from Steps 1-3. A real user in production has a purchase history that needs to be summarised into the same 29 features on the fly. The prediction function encapsulates this entire pipeline:

```
raw order history  -->  feature engineering  -->  imputation  -->  LightGBM  -->  probability
```

## Prerequisites
- Step 6 (LightGBM) must have been run -- `lgb_tuned.pkl` must exist in `../models/`
- Steps 1-3 artefacts must exist in `../prepared_data/`
- Virtual environment active with LightGBM installed


## 0. Imports & Configuration

In [ ]:
import os, warnings, joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

ARTEFACT_DIR = '../prepared_data'
MODEL_DIR    = '../models'
DATA_DIR     = '../data/instacart_data'

print(f'ARTEFACT_DIR : {os.path.abspath(ARTEFACT_DIR)}')
print(f'MODEL_DIR    : {os.path.abspath(MODEL_DIR)}')
print(f'DATA_DIR     : {os.path.abspath(DATA_DIR)}')


## 1. Load Model & Preprocessing Objects

In [ ]:
lgb_model    = joblib.load(os.path.join(MODEL_DIR,    'lgb_tuned.pkl'))
imputer      = joblib.load(os.path.join(ARTEFACT_DIR, 'imputer.pkl'))
FEATURE_COLS = joblib.load(os.path.join(ARTEFACT_DIR, 'feature_cols.pkl'))

print(f'Model loaded       : {type(lgb_model).__name__}')
print(f'Features expected  : {len(FEATURE_COLS)}')
print(f'Imputer loaded     : {type(imputer).__name__}')
print()
print('Feature list:')
for f in FEATURE_COLS:
    print(f'  {f}')


## 2. Load Raw Data

The prediction function needs access to the raw order history tables to compute features for any given (user, product) pair on the fly.


In [ ]:
import zipfile

# locate CSV files
csv_files = {}
for root, dirs, files in os.walk(DATA_DIR):
    for fname in files:
        if fname.endswith('.csv'):
            csv_files[fname] = os.path.join(root, fname)

orders               = pd.read_csv(csv_files['orders.csv'])
order_products_prior = pd.read_csv(csv_files['order_products__prior.csv'])
products             = pd.read_csv(csv_files['products.csv'])
aisles               = pd.read_csv(csv_files['aisles.csv'])
departments          = pd.read_csv(csv_files['departments.csv'])

products = (products
            .merge(aisles,       on='aisle_id',     how='left')
            .merge(departments,  on='department_id', how='left'))

prior_orders = orders[orders['eval_set'] == 'prior']
train_orders = orders[orders['eval_set'] == 'train']

# join prior line items with order metadata
prior_full = order_products_prior.merge(
    prior_orders[['order_id', 'user_id', 'order_number', 'order_dow',
                  'order_hour_of_day', 'days_since_prior_order']],
    on='order_id', how='left')

print(f'Raw data loaded.')
print(f'  orders               : {orders.shape}')
print(f'  order_products_prior : {order_products_prior.shape}')
print(f'  prior_full           : {prior_full.shape}')


## 3. The Prediction Function

The function `predict_reorder_probability(user_id, product_id, order_context)` takes three inputs:

| Input | Type | Description |
|---|---|---|
| `user_id` | int | The user whose history will be used |
| `product_id` | int | The product to score |
| `order_context` | dict | Timing features of the target order (dow, hour, days since prior) |

It returns a float between 0 and 1 representing the predicted reorder probability.

The function replicates the exact feature engineering pipeline from Steps 1-3, applied on the fly for a single (user, product) pair.


In [ ]:
def compute_user_features(user_id, prior_full):
    """Compute u_ features for a single user from their prior order history."""
    user_data = prior_full[prior_full['user_id'] == user_id]
    if user_data.empty:
        return None

    feats = {}
    feats['u_total_orders']      = user_data['order_id'].nunique()
    feats['u_total_products']    = len(user_data)
    feats['u_unique_products']   = user_data['product_id'].nunique()
    feats['u_total_reorders']    = user_data['reordered'].sum()
    feats['u_reorder_rate']      = user_data['reordered'].mean()
    feats['u_avg_basket_size']   = len(user_data) / user_data['order_id'].nunique()
    feats['u_avg_days_between']  = user_data['days_since_prior_order'].mean()
    feats['u_std_days_between']  = user_data['days_since_prior_order'].std()
    feats['u_avg_order_hour']    = user_data['order_hour_of_day'].mean()
    feats['u_avg_order_dow']     = user_data['order_dow'].mean()

    reordered_products = user_data[user_data['reordered'] == 1]['product_id'].nunique()
    feats['u_unique_reordered']       = reordered_products
    feats['u_unique_reorder_ratio']   = reordered_products / feats['u_unique_products']
    return feats


def compute_product_features(product_id, prior_full, products):
    """Compute p_ features for a single product from all prior orders."""
    prod_data = prior_full[prior_full['product_id'] == product_id]
    prod_meta = products[products['product_id'] == product_id]

    if prod_data.empty:
        # product has no prior order history -- use NaN, imputer will fill
        return {
            'p_total_purchases': np.nan, 'p_total_reorders': np.nan,
            'p_reorder_rate': np.nan,    'p_unique_users': np.nan,
            'p_avg_add_to_cart': np.nan, 'p_avg_order_hour': np.nan,
            'p_avg_order_dow': np.nan,
            'aisle_id': int(prod_meta['aisle_id'].values[0]) if not prod_meta.empty else np.nan,
            'department_id': int(prod_meta['department_id'].values[0]) if not prod_meta.empty else np.nan,
        }

    feats = {}
    feats['p_total_purchases']  = len(prod_data)
    feats['p_total_reorders']   = prod_data['reordered'].sum()
    feats['p_reorder_rate']     = prod_data['reordered'].mean()
    feats['p_unique_users']     = prod_data['user_id'].nunique()
    feats['p_avg_add_to_cart']  = prod_data['add_to_cart_order'].mean()
    feats['p_avg_order_hour']   = prod_data['order_hour_of_day'].mean()
    feats['p_avg_order_dow']    = prod_data['order_dow'].mean()

    if not prod_meta.empty:
        feats['aisle_id']      = int(prod_meta['aisle_id'].values[0])
        feats['department_id'] = int(prod_meta['department_id'].values[0])
    else:
        feats['aisle_id']      = np.nan
        feats['department_id'] = np.nan
    return feats


def compute_user_product_features(user_id, product_id, prior_full):
    """Compute up_ features for a (user, product) pair, excluding the most recent prior order."""
    user_data = prior_full[prior_full['user_id'] == user_id]
    if user_data.empty:
        return {'up_times_ordered': 0, 'up_reorder_rate': 0,
                'up_avg_add_to_cart': 0, 'up_orders_since_last': 0,
                'up_order_span': 0, 'up_in_last_order': 0}

    max_order_num    = user_data['order_number'].max()
    second_max       = user_data['order_number'].nlargest(2).iloc[-1] if len(user_data['order_number'].unique()) >= 2 else max_order_num

    # up_in_last_order: was the product in the most recent prior order?
    last_order_data  = user_data[user_data['order_number'] == max_order_num]
    up_in_last_order = int(product_id in last_order_data['product_id'].values)

    # behavioral features from history excluding the last order
    excl_last = user_data[
        (user_data['order_number'] < max_order_num) &
        (user_data['product_id'] == product_id)
    ]

    if excl_last.empty:
        return {'up_times_ordered': 0, 'up_reorder_rate': 0,
                'up_avg_add_to_cart': 0, 'up_orders_since_last': 0,
                'up_order_span': 0, 'up_in_last_order': up_in_last_order}

    last_order_num  = excl_last['order_number'].max()
    first_order_num = excl_last['order_number'].min()

    return {
        'up_times_ordered'   : len(excl_last),
        'up_reorder_rate'    : excl_last['reordered'].mean(),
        'up_avg_add_to_cart' : excl_last['add_to_cart_order'].mean(),
        'up_orders_since_last': second_max - last_order_num,
        'up_order_span'      : last_order_num - first_order_num,
        'up_in_last_order'   : up_in_last_order,
    }


def predict_reorder_probability(user_id, product_id, order_context,
                                 prior_full, products, model, imputer, feature_cols):
    """
    Predict the probability that user_id will reorder product_id.

    Parameters
    ----------
    user_id       : int   -- user to score
    product_id    : int   -- product to score
    order_context : dict  -- keys: t_order_dow (0-6), t_order_hour (0-23),
                             t_days_since_prior (float)
    prior_full    : DataFrame -- full prior order history
    products      : DataFrame -- product master with aisle/department
    model         : fitted LGBMClassifier
    imputer       : fitted SimpleImputer
    feature_cols  : list of feature column names

    Returns
    -------
    float : predicted reorder probability (0 to 1)
    """
    u_feats  = compute_user_features(user_id, prior_full)
    p_feats  = compute_product_features(product_id, prior_full, products)
    up_feats = compute_user_product_features(user_id, product_id, prior_full)

    if u_feats is None:
        raise ValueError(f'User {user_id} not found in prior order history.')

    row = {**u_feats, **p_feats, **up_feats, **order_context}

    # assemble into a single-row DataFrame in the exact feature order
    X = pd.DataFrame([row])[feature_cols]

    # impute any NaNs (products with no prior history)
    X_imp = pd.DataFrame(imputer.transform(X), columns=feature_cols)

    # apply categorical dtype for LightGBM
    for col in ['aisle_id', 'department_id']:
        X_imp[col] = X_imp[col].astype('category')

    prob = model.predict_proba(X_imp)[0, 1]
    return float(prob)


print('Prediction function defined successfully.')


## 4. Example Demonstrations

We demonstrate the prediction function on real users and products from the dataset, covering four scenarios:

- **High-frequency reorder:** A product the user orders very often
- **Occasional purchase:** A product ordered a few times but not recently
- **Never purchased:** A product the user has never ordered before
- **New user:** A user with very few prior orders


In [ ]:
# ── helper: describe a user-product pair ─────────────────────────────────────
def describe_pair(user_id, product_id, prior_full, products, train_orders):
    user_data = prior_full[(prior_full['user_id'] == user_id) &
                           (prior_full['product_id'] == product_id)]
    prod_name = products[products['product_id'] == product_id]['product_name']
    prod_name = prod_name.values[0] if not prod_name.empty else 'Unknown'
    n_orders  = prior_full[prior_full['user_id'] == user_id]['order_id'].nunique()
    n_times   = len(user_data)

    # did they actually reorder in the train set?
    train_order_ids = train_orders[train_orders['user_id'] == user_id]['order_id'].values
    from pathlib import Path
    import csv
    train_csv = [f for f in csv_files.values() if 'train' in f][0]
    train_products = pd.read_csv(train_csv)
    if len(train_order_ids) > 0:
        actual = int(product_id in
                     train_products[train_products['order_id'].isin(train_order_ids)]['product_id'].values)
    else:
        actual = None

    return prod_name, n_orders, n_times, actual


# ── find representative examples ──────────────────────────────────────────────
# load train products for ground truth
train_products_df = pd.read_csv([f for f in csv_files.values() if 'train' in f][0])

# Example 1: high-frequency reorder -- find user+product with many orders
freq_pairs = (prior_full.groupby(['user_id', 'product_id'])
              .size().reset_index(name='n_orders')
              .sort_values('n_orders', ascending=False))
ex1_user    = int(freq_pairs.iloc[0]['user_id'])
ex1_product = int(freq_pairs.iloc[0]['product_id'])

# Example 2: occasional purchase -- 2-4 orders, not in last order
occ_pairs = freq_pairs[(freq_pairs['n_orders'].between(2, 4))]
ex2_user    = int(occ_pairs.iloc[100]['user_id'])
ex2_product = int(occ_pairs.iloc[100]['product_id'])

# Example 3: never purchased -- pick a user and a product they have never ordered
ex3_user = ex1_user
user_products = set(prior_full[prior_full['user_id'] == ex3_user]['product_id'].unique())
all_products  = set(products['product_id'].unique())
never_ordered = list(all_products - user_products)
ex3_product   = never_ordered[0]

# Example 4: new user -- fewest prior orders
user_order_counts = prior_full.groupby('user_id')['order_id'].nunique()
new_user_id = int(user_order_counts[user_order_counts == user_order_counts.min()].index[0])
new_user_products = prior_full[prior_full['user_id'] == new_user_id]['product_id'].values
ex4_user    = new_user_id
ex4_product = int(new_user_products[0])

print('Examples identified:')
for label, uid, pid in [
    ('High-frequency', ex1_user, ex1_product),
    ('Occasional',     ex2_user, ex2_product),
    ('Never purchased',ex3_user, ex3_product),
    ('New user',       ex4_user, ex4_product),
]:
    pname = products[products['product_id'] == pid]['product_name'].values
    pname = pname[0] if len(pname) > 0 else 'Unknown'
    print(f'  {label:<20} user={uid}, product={pid} ({pname[:40]})')


In [ ]:
# ── default order context (midweek morning, ~7 days since last order) ─────────
default_context = {
    't_order_dow'        : 3,     # Wednesday
    't_order_hour'       : 10,    # 10am
    't_days_since_prior' : 7.0,   # one week since last order
}

examples = [
    ('High-frequency reorder',  ex1_user, ex1_product),
    ('Occasional purchase',     ex2_user, ex2_product),
    ('Never purchased before',  ex3_user, ex3_product),
    ('New user (few orders)',   ex4_user, ex4_product),
]

results = []
for scenario, uid, pid in examples:
    prob = predict_reorder_probability(
        user_id       = uid,
        product_id    = pid,
        order_context = default_context,
        prior_full    = prior_full,
        products      = products,
        model         = lgb_model,
        imputer       = imputer,
        feature_cols  = FEATURE_COLS
    )
    pname, n_orders, n_times, actual = describe_pair(uid, pid, prior_full, products, train_orders)
    results.append({
        'Scenario'         : scenario,
        'User ID'          : uid,
        'Product'          : pname[:35],
        'User total orders': n_orders,
        'Times bought'     : n_times,
        'Predicted prob'   : round(prob, 4),
        'Actual reorder'   : actual,
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


## 5. Sensitivity Analysis -- Effect of Order Context

The prediction function accepts order timing as an input. Here we test how sensitive the high-frequency example is to changes in the order context -- specifically `t_days_since_prior`, which captures how long since the user last ordered.


In [ ]:
days_range = [1, 3, 7, 14, 21, 30]
probs_by_day = []

for d in days_range:
    ctx = {'t_order_dow': 3, 't_order_hour': 10, 't_days_since_prior': float(d)}
    p = predict_reorder_probability(
        ex1_user, ex1_product, ctx, prior_full, products,
        lgb_model, imputer, FEATURE_COLS
    )
    probs_by_day.append(p)

plt.figure(figsize=(9, 4))
plt.plot(days_range, probs_by_day, 'o-', color='#DD8452', lw=2)
plt.axhline(0.5, linestyle=':', color='grey', label='Decision threshold (0.5)')
plt.xlabel('Days Since Prior Order')
plt.ylabel('Predicted Reorder Probability')
plt.title('Sensitivity to Days Since Prior Order (high-frequency user)')
plt.legend()
plt.tight_layout()
plt.show()

for d, p in zip(days_range, probs_by_day):
    print(f'  days_since_prior={d:>2}  -->  probability={p:.4f}')


## 6. Prediction Function Summary

The `predict_reorder_probability` function is the core inference unit for deployment. It:

- Accepts any (user_id, product_id, order_context) combination
- Replicates the full feature engineering pipeline from Steps 1-3 on the fly
- Handles edge cases: products never seen before (NaN imputation), new users   with minimal history
- Returns a calibrated probability between 0 and 1

### Expected outputs by scenario

| Scenario | Expected probability range | Reasoning |
|---|---|---|
| Product ordered 10+ times | 0.85 - 1.0 | Strong habitual purchase signal |
| Product ordered 2-4 times | 0.5 - 0.85 | Moderate history, uncertain |
| Product never ordered | 0.05 - 0.30 | No personal history; only product-level signal |
| New user (1-2 prior orders) | 0.30 - 0.60 | Limited history; relies on product averages |

### Next steps (Part 2)
Part 2 will build a **batch scoring function** that takes a single user_id and scores all products they have ever ordered, returning a ranked list sorted by reorder probability -- the core of a personalised reorder recommendation system.


In [ ]:
print('Step 8 Part 1 complete.')
print('Prediction function ready for use.')
print()
print('Function signature:')
print('  predict_reorder_probability(')
print('      user_id, product_id, order_context,')
print('      prior_full, products, model, imputer, feature_cols')
print('  ) -> float (probability 0-1)')
